In [14]:
import requests
import pandas as pd

def get_county_business_data(year='2022', state='*', county='*'):
    base_url = f'https://api.census.gov/data/{year}/cbp'
    
    params = {
        'get': 'GEO_ID,NAME,COUNTY,STATE,ESTAB,EMP,PAYANN,PAYQTR1',
        'for': f'county:{county}',
        'in': f'state:{state}',
        'NAICS2017': '00'  
    }
    
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        
        data = response.json()
        df = pd.DataFrame(data[1:], columns=data[0])
        
        numeric_cols = ['ESTAB', 'EMP', 'PAYANN', 'PAYQTR1']
        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        
        df.to_csv('county_business_data_2022.csv', index=False)
        return df
        
    except requests.exceptions.RequestException as e:
        print(f"Error making API request: {e}")
        return None

# Get data for all counties
df = get_county_business_data(year='2022')

In [15]:
sub_df = df.copy()

In [17]:
sub_df['fips_code'] = df['state'].astype(str) + df['county'].astype(str) 
sub_df = sub_df[['fips_code', 'ESTAB', 'NAME']]


In [18]:
sub_df

,fips_code,ESTAB,NAME
0,01001,948,"Autauga County, Alabama"
1,01003,6243,"Baldwin County, Alabama"
2,01005,421,"Barbour County, Alabama"
3,01007,310,"Bibb County, Alabama"
4,01009,767,"Blount County, Alabama"
...,...,...,...
3241,72151,185,"Yabucoa Municipio, Puerto Rico"
3242,72153,437,"Yauco Municipio, Puerto Rico"
3243,78010,920,"St. Croix Island, United States Virgin Islands"
3244,78020,245,"St. John Island, United States Virgin Islands"


In [19]:
housing = pd.read_csv("customers_served_county.csv", index_col=False)
housing


,fips_code,ESTAB,housing,customers_served
0,1001,948.0,24731.0,25679.0
1,1003,6243.0,128518.0,134761.0
2,1005,421.0,11702.0,12123.0
3,1007,310.0,9090.0,9400.0
4,1009,767.0,24793.0,25560.0
...,...,...,...,...
3243,72151,185.0,14165.0,14350.0
3244,72153,437.0,17199.0,17636.0
3245,78010,920.0,NaN,920.0
3246,78020,245.0,NaN,245.0


In [30]:
housing['fips_code'] = housing['fips_code'].astype(str).str.zfill(5)

df_merged = sub_df.merge(housing, on = 'fips_code', how = 'outer')

In [33]:
import numpy as np
len(np.unique(df_merged.fips_code.values))

3248

In [ ]:
df_merged
df_merged['customers_served'] = df_merged['ESTAB_x'].fillna(0) + df_merged['housing'].fillna(0)

df_merged.to_csv('customers_served_county_2022.csv', index=False)

KeyError: 'ESTAB'

In [23]:
housing = pd.read_csv("/Users/johnkim/Downloads/county_housing_units_2023.csv", index_col=False)
housing

,Geo_FIPS,Geo_QName,Geo_STUSAB,Geo_SUMLEV,Geo_GEOCOMP,Geo_US,Geo_REGION,Geo_DIVISION,Geo_STATE,Geo_COUNTY,...,Geo_SDUNI,Geo_UR,Geo_PCI,Geo_PUMA5,Geo_GEO_ID,Geo_NAME,Geo_BTTR,Geo_BTBG,Geo_PLACESE,SE_A10001_001
0,1001,"Autauga County, Alabama",al,50,0,NaN,NaN,NaN,1,1,...,NaN,NaN,NaN,NaN,0500000US01001,Autauga County,NaN,NaN,NaN,24731
1,1003,"Baldwin County, Alabama",al,50,0,NaN,NaN,NaN,1,3,...,NaN,NaN,NaN,NaN,0500000US01003,Baldwin County,NaN,NaN,NaN,128518
2,1005,"Barbour County, Alabama",al,50,0,NaN,NaN,NaN,1,5,...,NaN,NaN,NaN,NaN,0500000US01005,Barbour County,NaN,NaN,NaN,11702
3,1007,"Bibb County, Alabama",al,50,0,NaN,NaN,NaN,1,7,...,NaN,NaN,NaN,NaN,0500000US01007,Bibb County,NaN,NaN,NaN,9090
4,1009,"Blount County, Alabama",al,50,0,NaN,NaN,NaN,1,9,...,NaN,NaN,NaN,NaN,0500000US01009,Blount County,NaN,NaN,NaN,24793
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3217,72145,"Vega Baja Municipio, Puerto Rico",pr,50,0,NaN,NaN,NaN,72,145,...,NaN,NaN,NaN,NaN,0500000US72145,Vega Baja Municipio,NaN,NaN,NaN,25767
3218,72147,"Vieques Municipio, Puerto Rico",pr,50,0,NaN,NaN,NaN,72,147,...,NaN,NaN,NaN,NaN,0500000US72147,Vieques Municipio,NaN,NaN,NaN,5242
3219,72149,"Villalba Municipio, Puerto Rico",pr,50,0,NaN,NaN,NaN,72,149,...,NaN,NaN,NaN,NaN,0500000US72149,Villalba Municipio,NaN,NaN,NaN,9188
3220,72151,"Yabucoa Municipio, Puerto Rico",pr,50,0,NaN,NaN,NaN,72,151,...,NaN,NaN,NaN,NaN,0500000US72151,Yabucoa Municipio,NaN,NaN,NaN,14165


In [24]:
housing = housing[['Geo_FIPS', 'SE_A10001_001']].rename(columns = {'Geo_FIPS':'fips_code', 'SE_A10001_001':'housing'})
housing['fips_code'] = housing['fips_code'].astype(str).str.zfill(5)

df_merged = df.merge(housing, on = 'fips_code', how = 'outer')

In [27]:
df_merged['customers_served'] = df_merged['ESTAB'].fillna(0) + df_merged['housing'].fillna(0)

df_merged.to_csv('customers_served_county.csv', index=False)